# Projet #3 — Prédiction du Churn Client
## Notebook 2 — Modélisation, compréhension et validation

Objectif : montrer que l'on comprend **pourquoi** un modèle est utile, pas seulement afficher un score.

Dans ce notebook :
- on commence par un modèle naïf de référence ;
- on compare Régression Logistique, Arbre de Décision et Random Forest ;
- on analyse les erreurs avec la matrice de confusion ;
- on compare les performances train/test pour vérifier le sur-apprentissage ;
- on teste plusieurs seuils de décision (40 %, 50 %, 60 %) ;
- on termine par des limites et des recommandations métier simples.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
)

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/processed/telco_churn_clean.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Exécutez d'abord 01_exploration_preparation.ipynb pour créer "
        "data/processed/telco_churn_clean.csv"
    )

df = pd.read_csv(DATA_PATH)
print("Taille du dataset :", df.shape)
print("Taux de churn :", round(df["Churn"].mean() * 100, 2), "%")


## 1. Séparer les données

On utilise 80 % des données pour apprendre et 20 % pour tester.

Le jeu de test est gardé de côté jusqu'à la fin : il représente de nouvelles données que le modèle n'a jamais vues.


In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Train :", X_train.shape)
print("Test  :", X_test.shape)


## 2. Baseline naïve : pourquoi l'Accuracy ne suffit pas

Avant de faire du Machine Learning, on crée un modèle très simple qui prédit toujours la classe majoritaire : **pas de churn**.

Comme environ 73,5 % des clients ne churnent pas, ce modèle peut déjà avoir une Accuracy élevée.
Mais son Recall pour les churners est nul : il ne détecte aucun client qui part.

C'est une bonne démonstration du fait que l'Accuracy seule peut être trompeuse.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

pd.DataFrame([{
    "Modèle": "Baseline naïve",
    "Accuracy": accuracy_score(y_test, dummy_pred),
    "Precision": precision_score(y_test, dummy_pred, zero_division=0),
    "Recall": recall_score(y_test, dummy_pred, zero_division=0),
    "F1": f1_score(y_test, dummy_pred, zero_division=0),
}]).set_index("Modèle").round(4)


### À dire à l'oral

> Une Accuracy correcte ne veut pas forcément dire que le modèle est utile. La baseline naïve prédit presque toujours 'pas de churn', donc elle paraît correcte globalement mais elle ne détecte aucun churner.


## 3. Comparaison de trois modèles simples

On compare :
- Régression Logistique : simple et interprétable ;
- Arbre de Décision : facile à visualiser comme une suite de questions ;
- Random Forest : plusieurs arbres combinés pour obtenir un modèle plus robuste.


In [ ]:
models = {
    "Régression logistique": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Arbre de décision": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
}

def evaluate(name, model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    return pipe, {
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }

fitted_models = {}
rows = []
for name, model in models.items():
    fitted, metrics = evaluate(name, model)
    fitted_models[name] = fitted
    rows.append(metrics)

pd.DataFrame(rows).set_index("Modèle").round(4)


## 4. Analyse des erreurs avec la matrice de confusion

Une matrice de confusion permet de voir le type d'erreur commis par le modèle.

- Vrai positif : churn prédit et churn réel.
- Faux positif : churn prédit mais le client reste.
- Faux négatif : le modèle prédit qu'il reste mais le client churn.
- Vrai négatif : pas de churn prédit et pas de churn réel.

Dans notre cas, le faux négatif peut être coûteux : l'entreprise ne voit pas venir le départ du client.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, pipe) in zip(axes, fitted_models.items()):
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()


### À dire à l'oral

> Je regarde surtout les faux négatifs, car ce sont les clients réellement à risque que le modèle n'a pas détectés.


## 5. Vérifier le sur-apprentissage : train vs test

Un modèle peut être très bon sur les données qu'il connaît et moins bon sur de nouvelles données.

On compare donc le score sur le train et sur le test. Si l'écart est très grand, cela peut indiquer du sur-apprentissage (overfitting).


In [ ]:
comparison_rows = []
for name, pipe in fitted_models.items():
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)
    comparison_rows.append({
        "Modèle": name,
        "Accuracy train": accuracy_score(y_train, train_pred),
        "Accuracy test": accuracy_score(y_test, test_pred),
        "Recall train": recall_score(y_train, train_pred),
        "Recall test": recall_score(y_test, test_pred),
    })
pd.DataFrame(comparison_rows).set_index("Modèle").round(4)


## 6. Courbes ROC

La courbe ROC compare la capacité des modèles à distinguer churners et non-churners sur plusieurs seuils. Plus le ROC-AUC est élevé, meilleure est la capacité de classement.


In [ ]:
plt.figure(figsize=(8, 6))
ax = plt.gca()
for name, pipe in fitted_models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.title("Courbes ROC")
plt.tight_layout()
plt.show()


## 7. Petite optimisation avec GridSearchCV

On ne cherche pas des centaines de réglages. On teste seulement quelques valeurs faciles à expliquer.

GridSearchCV teste plusieurs réglages, utilise la validation croisée et garde celui qui donne le meilleur ROC-AUC moyen.


In [ ]:
searches = {
    "Régression logistique": (
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        {"model__C": [0.1, 1, 10]}
    ),
    "Arbre de décision": (
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        {"model__max_depth": [3, 5, 7], "model__min_samples_leaf": [10, 20, 40]}
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        {"model__n_estimators": [200, 300], "model__max_depth": [8, None], "model__min_samples_leaf": [1, 3]}
    ),
}

best_models = {}
grid_rows = []
for name, (model, params) in searches.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    grid = GridSearchCV(pipe, param_grid=params, scoring="roc_auc", cv=5, n_jobs=-1)
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_
    grid_rows.append({"Modèle": name, "ROC-AUC CV": grid.best_score_, "Meilleurs paramètres": grid.best_params_})

pd.DataFrame(grid_rows).set_index("Modèle")


## 8. Évaluation finale sur le jeu de test

Le test n'a pas servi au choix des hyperparamètres. On l'utilise seulement à la fin pour vérifier la capacité de généralisation.


In [ ]:
final_rows = []
for name, pipe in best_models.items():
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    final_rows.append({
        "Modèle": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })

pd.DataFrame(final_rows).set_index("Modèle").sort_values("ROC-AUC", ascending=False).round(4)


## 9. Comprendre le seuil de décision : 40 %, 50 %, 60 %

La régression logistique renvoie une probabilité de churn. Par défaut, une probabilité supérieure ou égale à 50 % est classée comme churn.

On teste 40 %, 50 % et 60 % pour comprendre le compromis entre Recall et Precision.


In [ ]:
log_model = best_models["Régression logistique"]
proba = log_model.predict_proba(X_test)[:, 1]

threshold_rows = []
for threshold in [0.40, 0.50, 0.60]:
    pred = (proba >= threshold).astype(int)
    threshold_rows.append({
        "Seuil": threshold,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "Clients signalés": int(pred.sum()),
    })

threshold_df = pd.DataFrame(threshold_rows).set_index("Seuil").round(4)
threshold_df


In [ ]:
threshold_df[["Precision", "Recall"]].plot(marker="o", figsize=(8, 5))
plt.title("Effet du seuil sur Precision et Recall")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


### À dire à l'oral

> Si je baisse le seuil à 40 %, je détecte davantage de clients à risque, donc le Recall augmente, mais je crée aussi plus de fausses alertes. Si je monte le seuil à 60 %, je signale moins de clients mais je risque de manquer davantage de churners.

Il n'existe donc pas un seuil parfait : il dépend du coût métier des erreurs.


## 10. Recommandation simple

Pour ce projet étudiant, la **Régression Logistique** reste le modèle principal recommandé :
- ses performances sont proches de la Random Forest ;
- elle est plus simple à interpréter ;
- elle permet de produire une probabilité ;
- son seuil peut être adapté selon la stratégie de rétention.


## 11. Recommandations métier simples

Les analyses du projet montrent notamment des associations avec l'ancienneté, le type de contrat, les contrats mensuels, la fibre optique et les frais mensuels/totaux.

Une entreprise pourrait utiliser le modèle pour prioriser les clients à contacter :
1. le modèle calcule un risque de churn ;
2. les clients les plus à risque sont placés dans une liste ;
3. l'équipe rétention les contacte avec une offre ou une enquête de satisfaction.

Important : ces variables sont associées au churn dans les données. Elles ne prouvent pas une cause.


## 12. Limites du projet

1. Les données sont historiques : les comportements peuvent changer.
2. Le dataset ne contient pas toutes les raisons personnelles d'un départ.
3. Le modèle détecte des associations, pas les causes réelles du churn.
4. Le seuil métier idéal n'est pas défini sans connaître le coût des erreurs.
5. En production, il faudrait vérifier et réentraîner le modèle régulièrement.


## 13. Ce que j'ai appris

Ce projet m'a permis de comprendre que :
- nettoyer les données est essentiel ;
- une bonne Accuracy ne suffit pas si les classes sont déséquilibrées ;
- il faut séparer train et test pour évaluer la généralisation ;
- plusieurs métriques sont nécessaires pour comprendre les erreurs ;
- un modèle simple peut être préférable à un modèle plus complexe si les performances sont proches ;
- le choix du seuil dépend du besoin métier ;
- Machine Learning ne veut pas dire chercher le modèle le plus compliqué.


## Conclusion

Le projet suit une démarche complète mais simple : préparation, baseline, comparaison des modèles, analyse des erreurs, vérification du sur-apprentissage, petite optimisation, test de seuils et recommandation métier.

Le modèle principal retenu est la **Régression Logistique**, car elle offre un bon compromis entre performance, détection des churners et simplicité d'interprétation.
